# RL Temel Kavramlar: On-Policy, Off-Policy ve Diğer Ayrımlar

Bu notebook diğer tutorialların altyapısını kurar. DQN, REINFORCE, Actor-Critic, A2C/A3C ve PPO'yu daha doğru konumlandırmak için temel ayrımları açıklar.

## 1. Behavior Policy ve Target Policy

RL algoritmalarını anlamak için iki politika ayrımı önemlidir:

| Kavram | Anlam |
| --- | --- |
| Behavior policy $\mu(a\mid s)$ | Veriyi ortamdan toplayan politika |
| Target policy $\pi(a\mid s)$ | Öğrenmek/iyileştirmek istediğimiz politika |

Eğer veri toplayan politika ile öğrenilen politika aynıysa yöntem on-policy'dir. Farklı olabiliyorlarsa off-policy'dir.

## 2. On-Policy Nedir?

On-policy algoritmalarda veri, güncellenen politikanın kendisinden veya ona çok yakın eski bir sürümünden gelir:

$$\mu \approx \pi_\theta$$

Örnekler:

- REINFORCE
- Actor-Critic'in klasik halleri
- A2C / A3C
- PPO

On-policy yöntemlerde rollout eskidikçe değerini kaybeder. Çünkü politika güncellendikten sonra eski aksiyon olasılıkları artık yeni politikayı tam temsil etmez.

Pratik sonucu: On-policy yöntemler genellikle replay buffer'ı DQN gibi uzun süreli kullanmaz. Toplanan rollout birkaç epoch kullanılır ve sonra atılır veya yenilenir.

## 3. Off-Policy Nedir?

Off-policy algoritmalarda veri toplayan politika ile öğrenilen politika farklı olabilir:

$$\mu \neq \pi$$

DQN bunun klasik örneğidir. Ajan ortamda epsilon-greedy davranır:

$$\mu(a\mid s)=\epsilon\text{-greedy}(Q_\theta)$$

Ama öğrenilen hedef greedy optimal politikadır:

$$\pi(s)=\arg\max_a Q(s,a)$$

Q-learning hedefinde davranış politikasının seçtiği sonraki aksiyon değil, sonraki durumdaki maksimum Q değeri kullanılır:

$$y=r+\gamma\max_{a'}Q(s',a')$$

Bu yüzden DQN eski replay örneklerinden öğrenebilir. Veri epsilon-greedy eski bir ağdan gelmiş olsa bile hedef, greedy optimal Q fonksiyonuna yaklaşır.

## 4. On-Policy ve Off-Policy Karşılaştırması

| Özellik | On-policy | Off-policy |
| --- | --- | --- |
| Veri kaynağı | Mevcut veya çok yakın politika | Farklı/eski behavior policy olabilir |
| Replay kullanımı | Sınırlı | Doğal ve yaygın |
| Sample efficiency | Genelde daha düşük | Genelde daha yüksek |
| Stabilite | Daha doğrudan ama veri pahalı | Replay/target/correction gerekebilir |
| Örnek algoritmalar | REINFORCE, A2C, PPO | Q-learning, DQN, Double DQN |
| Orbit Wars etkisi | Sık rollout toplamak gerekir | Eski self-play verisi kullanılabilir ama stale olabilir |

Self-play ortamlarında off-policy replay daha karmaşık hale gelir. Çünkü sadece ajan değil, rakip dağılımı da zamanla değişir. Çok eski replay verisi artık güncel rakip stratejilerini temsil etmeyebilir.

## 5. PPO Tam Olarak On-Policy mi?

PPO pratikte on-policy kabul edilir, ama aynı rollout üzerinde birkaç epoch minibatch update yapar. Bunu güvenli tutmak için eski politika olasılığı saklanır:

$$r_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{old}(a_t\mid s_t)}$$

Clipping, yeni politikanın rollout'u toplayan eski politikadan çok uzaklaşmasını engeller:

$$clip(r_t,1-\epsilon,1+\epsilon)$$

Bu yüzden PPO, DQN gibi uzun süreli replay kullanan off-policy bir yöntem değildir. Sadece kısa süreli veri tekrar kullanımına izin veren on-policy bir actor-critic yöntemi gibi düşünülmelidir.

## 6. Bootstrap Nedir?

Bootstrap, hedefi gerçek tam return yerine modelin kendi sonraki tahminiyle kurmaktır.

DQN hedefi:

$$y=r+\gamma\max_{a'}Q_{\theta^-}(s',a')$$

Actor-Critic value hedefi:

$$y_V=r+\gamma(1-d)V_\phi(s')$$

Bootstrap daha sık ve düşük varyanslı güncelleme sağlar, fakat tahmin hatası hedefe de girdiği için bias ekleyebilir. Monte Carlo return ise bootstrap kullanmaz; daha az bias, daha yüksek varyans üretir.

## 7. Model-Free ve Model-Based

| Ayrım | Anlam |
| --- | --- |
| Model-free | Ortam geçiş modelini öğrenmeden veya kullanmadan politika/değer öğrenir |
| Model-based | Ortam dinamiğini $P(s'\mid s,a)$ veya bir simülatör modeliyle planlama yapar |

DQN, PPO, REINFORCE ve Actor-Critic burada model-free yöntemlerdir. Orbit Wars ortamı simülatör olarak kullanılır, ama algoritma açık bir geçiş modeli öğrenip planlama yapmaz.

## 8. Value-Based, Policy-Based ve Actor-Critic

| Aile | Ne öğrenir? | Örnek |
| --- | --- | --- |
| Value-based | $Q(s,a)$ veya $V(s)$ | Q-learning, DQN, Double/Dueling DQN |
| Policy-based | $\pi(a\mid s)$ | REINFORCE |
| Actor-Critic | Hem politika hem value | A2C, A3C, PPO |

DQN aksiyonu Q değerlerinden greedy seçer. Policy gradient yöntemleri aksiyon dağılımını doğrudan üretir. Actor-Critic, politika öğrenirken critic'i baseline/advantage üretmek için kullanır.

## 8.1. Minimal Ağ Aileleri

Aşağıdaki hücre üç temel neural network ailesini yan yana gösterir:

- Value-based ağ: her aksiyon için $Q(s,a)$ üretir.
- Policy-based ağ: aksiyon dağılımı için logits üretir.
- Actor-Critic ağ: aynı temsil üzerinden hem politika hem value üretir.

Bu hücre Orbit Wars mimarisinin küçük MLP karşılığıdır; gerçek repoda bu MLP trunk yerine Set Transformer encoder kullanılır.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu hücre için PyTorch gerekli. Jupyter'da orbit_wars_env kernelini seçin.") from exc

class ValueBasedQNet(nn.Module):
    """Value-based aile: durumdan her ayrık aksiyon için Q değeri üretir."""
    def __init__(self, obs_dim=12, hidden_dim=64, n_actions=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, obs):
        return self.net(obs)  # (B, n_actions)

class PolicyNet(nn.Module):
    """Policy-based aile: durumdan stokastik politika logits'i üretir."""
    def __init__(self, obs_dim=12, hidden_dim=64, n_actions=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, obs):
        return self.net(obs)  # Categorical(logits=...) ile dağılıma çevrilir

class ActorCriticNet(nn.Module):
    """Actor-Critic aile: ortak trunk + policy head + value head."""
    def __init__(self, obs_dim=12, hidden_dim=64, n_actions=5):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(obs_dim, hidden_dim), nn.Tanh())
        self.policy_head = nn.Linear(hidden_dim, n_actions)
        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self, obs):
        h = self.trunk(obs)
        return self.policy_head(h), self.value_head(h).squeeze(-1)

obs = torch.randn(3, 12)
q_values = ValueBasedQNet()(obs)
policy_logits = PolicyNet()(obs)
ac_logits, ac_values = ActorCriticNet()(obs)

print("Q-values:", q_values.shape)
print("Policy logits:", policy_logits.shape)
print("Actor-Critic logits/value:", ac_logits.shape, ac_values.shape)

## 8.2. Bu Ağlar Nasıl Update Edilir?

Ağ ailesi değiştikçe update kuralı da değişir:

- Value-based: seçilen aksiyonun Q değeri TD target'a yaklaştırılır.
- Policy-based: seçilen aksiyonun log olasılığı return/advantage ile ağırlıklandırılır.
- Actor-Critic: policy loss ve value loss birlikte optimize edilir.

Aşağıdaki hücre üç update tipini minimal şekilde yan yana gösterir.

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Categorical

obs = torch.randn(4, 12)
next_obs = torch.randn(4, 12)
actions = torch.tensor([0, 1, 2, 3])
rewards = torch.tensor([1.0, 0.0, -0.5, 0.2])
dones = torch.tensor([0.0, 0.0, 1.0, 0.0])
gamma = 0.99

# 1) Value-based DQN tipi update.
q_net = ValueBasedQNet(obs_dim=12, n_actions=5)
target_q_net = ValueBasedQNet(obs_dim=12, n_actions=5)
target_q_net.load_state_dict(q_net.state_dict())
q_optim = torch.optim.Adam(q_net.parameters(), lr=1e-3)

q_taken = q_net(obs).gather(1, actions[:, None]).squeeze(1)
with torch.no_grad():
    td_target = rewards + gamma * (1.0 - dones) * target_q_net(next_obs).max(dim=1).values
q_loss = F.smooth_l1_loss(q_taken, td_target)
q_optim.zero_grad(set_to_none=True)
q_loss.backward()
q_optim.step()

# 2) Policy-gradient REINFORCE tipi update.
policy_net = PolicyNet(obs_dim=12, n_actions=5)
policy_optim = torch.optim.Adam(policy_net.parameters(), lr=1e-3)
returns = torch.tensor([1.0, 0.4, -0.2, 0.8])
dist = Categorical(logits=policy_net(obs))
log_probs = dist.log_prob(actions)
pg_loss = -(log_probs * returns).mean()
policy_optim.zero_grad(set_to_none=True)
pg_loss.backward()
policy_optim.step()

# 3) Actor-Critic tipi update.
ac_net = ActorCriticNet(obs_dim=12, n_actions=5)
ac_optim = torch.optim.Adam(ac_net.parameters(), lr=1e-3)
logits, values = ac_net(obs)
with torch.no_grad():
    _, next_values = ac_net(next_obs)
    value_target = rewards + gamma * (1.0 - dones) * next_values
advantages = value_target - values
dist = Categorical(logits=logits)
ac_policy_loss = -(dist.log_prob(actions) * advantages.detach()).mean()
ac_value_loss = 0.5 * advantages.pow(2).mean()
ac_loss = ac_policy_loss + ac_value_loss
ac_optim.zero_grad(set_to_none=True)
ac_loss.backward()
ac_optim.step()

print({
    "q_loss": float(q_loss.detach()),
    "pg_loss": float(pg_loss.detach()),
    "ac_loss": float(ac_loss.detach()),
})

## 9. Orbit Wars İçin Hızlı Sınıflandırma

| Notebook / yöntem | On-policy mi? | Off-policy mi? | Bootstrap? | Ana öğrenilen şey |
| --- | --- | --- | --- | --- |
| DQN | Hayır | Evet | Evet | $Q(s,a)$ |
| Double DQN | Hayır | Evet | Evet | Daha az overestimate bias'lı $Q(s,a)$ |
| Dueling DQN | Hayır | Evet | Evet | Value + advantage ayrıştırmalı $Q(s,a)$ |
| REINFORCE | Evet | Hayır | Hayır | $\pi(a\mid s)$ |
| Actor-Critic | Evet | Genelde hayır | Evet | $\pi(a\mid s)$ ve $V(s)$ |
| A2C/A3C | Evet | Hayır | Evet | Paralel rollout'lu actor-critic |
| PPO | Evet | Uzun replay anlamında hayır | Evet | Clipped actor-critic |

Bu tablo pratik sınıflandırmadır. Literatürde off-policy actor-critic yöntemleri de vardır, fakat bu klasördeki PPO/A2C çizgisi on-policy olarak ele alınır.